# Code 2.6.5: Training on XOR

Trains a network with four hidden units on the four XOR points using the functions of Code 2.6.4, logging the loss every 100 steps.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 2.6.1
import math

# From Code 2.6.4
import random

# From Code 2.6.1
class Scalar:
    __slots__ = ("value", "grad", "inputs", "op")

    def __init__(self, value, inputs=(), op=""):
        self.value = float(value)
        self.grad = 0.0
        self.inputs = inputs      # tuple of (parent Scalar, d self / d parent)
        self.op = op              # operation name, for printing and debugging

    def __repr__(self):
        return f"Scalar({self.value:.4f}, grad={self.grad:.4f}, op='{self.op}')"

    # ---- building blocks -------------------------------------------------
    @staticmethod
    def lift(x):
        return x if isinstance(x, Scalar) else Scalar(x)

    def __add__(self, other):
        other = Scalar.lift(other)
        return Scalar(self.value + other.value, ((self, 1.0), (other, 1.0)), "+")

    def __mul__(self, other):
        other = Scalar.lift(other)
        return Scalar(self.value * other.value,
                      ((self, other.value), (other, self.value)), "*")

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents"
        return Scalar(self.value ** k, ((self, k * self.value ** (k - 1)),), f"^{k}")

    def exp(self):
        e = math.exp(self.value)
        return Scalar(e, ((self, e),), "exp")

    def log(self):
        return Scalar(math.log(self.value), ((self, 1.0 / self.value),), "log")

    def tanh(self):
        t = math.tanh(self.value)
        return Scalar(t, ((self, 1.0 - t * t),), "tanh")

    def relu(self):
        return Scalar(max(0.0, self.value), ((self, float(self.value > 0)),), "relu")

    # ---- derived operations (built from the ones above) ------------------
    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-Scalar.lift(other))

    def __truediv__(self, other):
        return self * Scalar.lift(other) ** -1

    __radd__ = __add__
    __rmul__ = __mul__

    def __rsub__(self, other):
        return Scalar.lift(other) - self

    def __rtruediv__(self, other):
        return Scalar.lift(other) / self

    # ---- reverse pass -------------------------------------------------------
    def backward(self):
        """Fill in .grad for every Scalar that this one depends on."""
        order, visited = [], set()
        stack = [(self, False)]
        while stack:                       # iterative depth-first topological sort
            node, children_done = stack.pop()
            if children_done:
                order.append(node)
                continue
            if id(node) in visited:
                continue
            visited.add(id(node))
            stack.append((node, True))
            for parent, _ in node.inputs:
                if id(parent) not in visited:
                    stack.append((parent, False))
        self.grad = 1.0
        for node in reversed(order):       # outputs before inputs
            for parent, local in node.inputs:
                parent.grad += local * node.grad

# From Code 2.6.4
def make_params(n_in, n_hidden, seed=0):
    rng = random.Random(seed)
    W1 = [[Scalar(rng.gauss(0, 1 / math.sqrt(n_in))) for _ in range(n_hidden)] for _ in range(n_in)]
    b1 = [Scalar(0.0) for _ in range(n_hidden)]
    W2 = [Scalar(rng.gauss(0, 1 / math.sqrt(n_hidden))) for _ in range(n_hidden)]
    b2 = Scalar(0.0)
    return W1, b1, W2, b2

def all_params(params):
    W1, b1, W2, b2 = params
    return [w for row in W1 for w in row] + b1 + W2 + [b2]

def logit(params, x):
    """Forward pass for one example x (a list of floats); returns a Scalar logit."""
    W1, b1, W2, b2 = params
    h = []
    for j in range(len(b1)):
        z = b1[j]
        for i in range(len(x)):
            z = z + x[i] * W1[i][j]
        h.append(z.tanh())
    out = b2
    for j in range(len(h)):
        out = out + h[j] * W2[j]
    return out

def bce_from_logit(z, y):
    """Binary cross-entropy -[y log σ(z) + (1-y) log(1-σ(z))], computed stably.

    Uses log(1 + e^z) - y z, rewritten as z + log(1 + e^-z) - y z when z > 0
    so that exp never receives a large positive argument.
    """
    if z.value > 0:
        return z + ((-z).exp() + 1).log() - y * z
    return (z.exp() + 1).log() - y * z

def train(X, Y, n_hidden=8, lr=0.5, steps=200, seed=0, log=None):
    params = make_params(len(X[0]), n_hidden, seed)
    history = []
    for step in range(steps):
        loss = sum((bce_from_logit(logit(params, x), y) for x, y in zip(X, Y)), Scalar(0.0))
        loss = loss * (1.0 / len(X))
        for p in all_params(params):
            p.grad = 0.0                 # 1. zero old gradients
        loss.backward()                  # 2. backpropagate
        for p in all_params(params):
            p.value -= lr * p.grad       # 3. gradient-descent step
        history.append(loss.value)
        if log and step % log == 0:
            print(f"step {step:4d}  loss {loss.value:.4f}")
    return params, history


In [ ]:
X_xor = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
Y_xor = [0, 1, 1, 0]
params, history = train(X_xor, Y_xor, n_hidden=4, lr=1.0, steps=300, seed=3, log=100)
probs = [1 / (1 + math.exp(-logit(params, x).value)) for x in X_xor]
print("predicted P(y=1):", [round(q, 3) for q in probs])


**Expected output**

Output:

```text
step    0  loss 0.7330
step  100  loss 0.0592
step  200  loss 0.0155
predicted P(y=1): [0.003, 0.988, 0.991, 0.011]
```
